# TalkNet-ASD: Optimized Inference Demo
**GPU Required:** `Runtime -> Change runtime type -> T4 GPU`, then Run All.

In [ ]:
# GPU GUARD: Fail immediately if no GPU is attached
import torch
if not torch.cuda.is_available():
    raise RuntimeError(
        "\n\n" + "="*60 + "\n"
        "NO GPU DETECTED!\n"
        "Go to: Runtime -> Change runtime type -> T4 GPU -> Save\n"
        "Then: Runtime -> Run All\n"
        + "="*60
    )
print(f"GPU OK: {torch.cuda.get_device_name(0)}")

# Cell 1: Setup Environment
import os
os.chdir("/content")

!rm -rf TalkNet-ASD
!git clone -q https://github.com/TaoRuijie/TalkNet-ASD.git
os.chdir("/content/TalkNet-ASD")

!pip install -q scenedetect==0.5.6.1
!pip install -q gdown scipy librosa opencv-python python_speech_features tqdm
!apt-get update -qq && apt-get install -y -qq ffmpeg > /dev/null 2>&1

# Patch deprecated np.int/np.float for NumPy 1.24+
!find . -name "*.py" -exec sed -i "s/np\.int)/int)/g" {} +
!find . -name "*.py" -exec sed -i "s/np\.int,/int,/g" {} +
!find . -name "*.py" -exec sed -i "s/np\.int]/int]/g" {} +
!find . -name "*.py" -exec sed -i "s/np\.float)/float)/g" {} +
!find . -name "*.py" -exec sed -i "s/np\.bool)/bool)/g" {} +

print("=== Setup complete ===")

In [ ]:
# Cell 2: Upload & Pre-process Video
# We:
#   1. Accept your upload
#   2. Resize to 480p and limit to 60 seconds using ffmpeg BEFORE passing to TalkNet
#      This is the single biggest speedup: less data = faster everything downstream.
import os, time
from google.colab import files
os.chdir("/content/TalkNet-ASD")
!mkdir -p demo

print("Upload your video (mp4):")
uploaded = files.upload()
filename = list(uploaded.keys())[0]
video_name_no_ext = "test_clip"

print(f"\nPre-processing {filename} (resize to 480p, resize to 480p for faster inference)...")
t0 = time.time()

# Key optimizations:
# -t 60         : take only first 60 seconds
# -vf scale=-2:480 : resize to 480p (preserving aspect ratio)
# -crf 23       : fast encode
# -preset fast  : fast encode
# -y            : overwrite without prompting
!ffmpeg -y -i "{filename}" -vf scale=-2:480 -crf 23 -preset fast \
    demo/test_clip.mp4 2>&1 | tail -5

elapsed = time.time() - t0
size = os.path.getsize("demo/test_clip.mp4") / (1024*1024)
print(f"\nPre-processing done in {elapsed:.1f}s => demo/test_clip.mp4 ({size:.1f} MB)")
print("=== Video ready ===")

In [ ]:
# Cell 3: Run Optimized TalkNet Inference
# Tuned flags for maximum speed:
#   --nDataLoaderThread 4  : 4 parallel workers (T4 GPU has 4 cores)
#   --facedetScale 0.25    : already default, keep it (bigger = slower)
#   --minTrack 5           : from 10 -> 5 (accept shorter face tracks, fewer misses)
#   --numFailedDet 20      : allow more misses before dropping a track (avoids re-detection)
import os, time
os.chdir("/content/TalkNet-ASD")

print(f"Running TalkNet inference (optimized)...")
t0 = time.time()

!python demoTalkNet.py \
    --videoName test_clip \
    --nDataLoaderThread 4 \
    --facedetScale 0.25 \
    --minTrack 5 \
    --numFailedDet 20 \

elapsed = time.time() - t0
print(f"\n=== Inference done in {elapsed:.0f}s ({elapsed/60:.1f} min) ===")

In [ ]:
# Cell 4: Download Result
import os
os.chdir("/content/TalkNet-ASD")
from google.colab import files

output_path = "demo/test_clip/pyavi/video_out.avi"

if os.path.exists(output_path):
    size = os.path.getsize(output_path) / (1024*1024)
    print(f"SUCCESS: {output_path} ({size:.1f} MB)")
    files.download(output_path)
else:
    print("Not found. Full demo tree:")
    !ls -R demo/

---
# Task 00012: Standalone Face-Crop Preprocessing
Run cells 5–6 to test `preprocess_faces.py` in isolation, **without** running the TalkNet active speaker model.
This validates the face detection + tracking + crop pipeline independently.

In [ ]:
# Cell 5: Upload preprocess_faces.py and run standalone preprocessing
import os, json
os.chdir("/content/TalkNet-ASD")

print("Upload preprocess_faces.py from your scripts/ml/ folder:")
from google.colab import files
uploaded_script = files.upload()

# Move script into the TalkNet repo root so it can import model.faceDetector.s3fd
!mv preprocess_faces.py /content/TalkNet-ASD/

# Apply the same NumPy compat patches to the script itself
!sed -i "s/np\.int)/int)/g" preprocess_faces.py
!sed -i "s/numpy\.int)/int)/g" preprocess_faces.py

print("\n=== Running Standalone Face-Crop Preprocessing ===")
print("Input: demo/test_clip.mp4")
print("Output: demo/preprocessing_output/")

!python preprocess_faces.py \
    --videoPath demo/test_clip.mp4 \
    --savePath demo/preprocessing_output \
    --nDataLoaderThread 4 \
    --minTrack 5 \
    --numFailedDet 20


In [ ]:
# Cell 6: Inspect metadata.json and verify outputs
import os, json
os.chdir("/content/TalkNet-ASD")

meta_path = "demo/preprocessing_output/metadata.json"

if os.path.exists(meta_path):
    with open(meta_path) as f:
        meta = json.load(f)

    print(f"SUCCESS: {len(meta[\"tracks\"])} face tracks found")
    print("\n--- metadata.json (first 2 tracks) ---")
    preview = {"tracks": []}
    for t in meta["tracks"][:2]:
        # Only show first 3 bbox frames for readability
        t_preview = dict(t)
        t_preview["bbox_history"] = t["bbox_history"][:3]
        preview["tracks"].append(t_preview)
    print(json.dumps(preview, indent=2))

    print("\n--- Cropped face videos in pycrop/ ---")
    !ls -lh demo/preprocessing_output/pycrop/*.avi

    print("\n--- Downloading metadata.json ---")
    from google.colab import files
    files.download(meta_path)
else:
    print("NOT FOUND. Full output tree:")
    !ls -R demo/preprocessing_output/


---
# Cell 7: Automated Test Suite
Validates every aspect of the preprocessing output: directory structure, metadata schema, video/audio integrity, and cross-file coverage.

In [ ]:
# Cell 7: Run test_preprocess_faces.py validation suite
import os
os.chdir("/content/TalkNet-ASD")

print("Upload test_preprocess_faces.py from your scripts/ml/ folder:")
from google.colab import files
uploaded = files.upload()
!mv test_preprocess_faces.py /content/TalkNet-ASD/

print("\n=== Running Validation Suite ===")
!python test_preprocess_faces.py --outputDir demo/preprocessing_output


---
# Cell 8: Visual Inspection — What Did the Model Actually Crop?
Renders a frame grid of each detected face track so you can visually verify the crops are correct.

In [ ]:
# Cell 8: Visual inspection of cropped face tracks
import os, cv2, glob
import numpy as np
import matplotlib.pyplot as plt
import json
os.chdir("/content/TalkNet-ASD")

OUTPUT_DIR = "demo/preprocessing_output"
CROP_DIR   = os.path.join(OUTPUT_DIR, "pycrop")
META_PATH  = os.path.join(OUTPUT_DIR, "metadata.json")
SAMPLES_PER_TRACK = 5  # frames to sample from each track

with open(META_PATH) as f:
    meta = json.load(f)

avi_files = sorted(glob.glob(os.path.join(CROP_DIR, "*.avi")))
n_tracks  = len(avi_files)

fig, axes = plt.subplots(
    n_tracks, SAMPLES_PER_TRACK,
    figsize=(SAMPLES_PER_TRACK * 2.5, n_tracks * 2.5)
)
# Handle single-track edge case
if n_tracks == 1:
    axes = [axes]

for row, avi_path in enumerate(avi_files):
    track_name = os.path.splitext(os.path.basename(avi_path))[0]
    cap = cv2.VideoCapture(avi_path)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    # Sample evenly-spaced frames
    sample_indices = np.linspace(0, total_frames - 1, SAMPLES_PER_TRACK, dtype=int)

    # Find matching track in metadata for time info
    track_meta = next((t for t in meta["tracks"] if t["track_id"] == track_name), {})
    start_t = track_meta.get("start_time_sec", 0)
    end_t   = track_meta.get("end_time_sec", 0)

    for col, frame_idx in enumerate(sample_indices):
        cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
        ret, frame = cap.read()
        ax = axes[row][col]
        if ret:
            ax.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        else:
            ax.text(0.5, 0.5, "N/A", ha="center", va="center")
        ax.axis("off")
        if col == 0:
            ax.set_title(f"Track {track_name}\n{start_t:.1f}s–{end_t:.1f}s",
                         fontsize=8, loc="left")
        else:
            ax.set_title(f"frame {frame_idx}", fontsize=7)
    cap.release()

plt.suptitle(
    f"Face Crop Visual Inspection — {n_tracks} Tracks Detected\n"
    f"Each row = one face track  |  Each column = sampled frame",
    fontsize=11, y=1.02
)
plt.tight_layout()
plt.savefig("demo/preprocessing_output/crop_visual_check.png",
            dpi=120, bbox_inches="tight")
plt.show()
print(f"\nSaved grid to demo/preprocessing_output/crop_visual_check.png")
from google.colab import files
files.download("demo/preprocessing_output/crop_visual_check.png")


---
# Cell 9: Re-run with a Two-Speaker Webinar Video (Task 00012)
The earlier test used a single-speaker podcast. Task 00012 requires a **two-speaker** sample.
Upload a webinar/interview video with two visible speakers so each person gets their own face tracks.

In [ ]:
# Cell 9: Run preprocessing on a two-speaker webinar clip
# Upload any interview/webinar video where TWO people are visible (side-by-side or alternating)
# e.g. a Zoom call recording, a TV interview, a podcast with two cameras showing both speakers
import os, time
from google.colab import files
os.chdir("/content/TalkNet-ASD")
!mkdir -p demo

print("Upload a two-speaker webinar/interview video (.mp4):")
print("(Tip: any video where TWO faces are clearly visible works)")
uploaded = files.upload()
filename = list(uploaded.keys())[0]

# Resize to 480p for speed (keeps both faces clearly visible)
print(f"\nPre-processing {filename} (resize to 480p)...")
!ffmpeg -y -i "{filename}" -vf scale=-2:480 -crf 23 -preset fast \
    demo/two_speaker_clip.mp4 2>&1 | tail -3

print("\n=== Running Preprocessing on Two-Speaker Clip ===")
t0 = time.time()
!python preprocess_faces.py \
    --videoPath demo/two_speaker_clip.mp4 \
    --savePath demo/two_speaker_output \
    --nDataLoaderThread 4 \
    --minTrack 5 \
    --numFailedDet 20
elapsed = time.time() - t0
print(f"\nDone in {elapsed:.0f}s")


In [ ]:
# Cell 10: Visual check — two speakers should show as distinct faces in different rows
import os, cv2, glob, json
import numpy as np
import matplotlib.pyplot as plt
os.chdir("/content/TalkNet-ASD")

OUTPUT_DIR = "demo/two_speaker_output"
CROP_DIR   = os.path.join(OUTPUT_DIR, "pycrop")
META_PATH  = os.path.join(OUTPUT_DIR, "metadata.json")
SAMPLES_PER_TRACK = 4

with open(META_PATH) as f:
    meta = json.load(f)

avi_files = sorted(glob.glob(os.path.join(CROP_DIR, "*.avi")))
n_tracks  = len(avi_files)
print(f"Found {n_tracks} face tracks — if >1 speaker, you should see different faces per row")

fig, axes = plt.subplots(
    n_tracks, SAMPLES_PER_TRACK,
    figsize=(SAMPLES_PER_TRACK * 2.8, n_tracks * 2.8)
)
if n_tracks == 1:
    axes = [axes]

for row, avi_path in enumerate(avi_files):
    track_name  = os.path.splitext(os.path.basename(avi_path))[0]
    cap         = cv2.VideoCapture(avi_path)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    sample_idx  = np.linspace(0, total_frames - 1, SAMPLES_PER_TRACK, dtype=int)
    track_meta  = next((t for t in meta["tracks"] if t["track_id"] == track_name), {})
    start_t     = track_meta.get("start_time_sec", 0)
    end_t       = track_meta.get("end_time_sec", 0)

    for col, fidx in enumerate(sample_idx):
        cap.set(cv2.CAP_PROP_POS_FRAMES, fidx)
        ret, frame = cap.read()
        ax = axes[row][col] if n_tracks > 1 else axes[col]
        if ret:
            ax.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        ax.axis("off")
        ax.set_title(
            f"Track {track_name} ({start_t:.1f}s–{end_t:.1f}s)" if col == 0 else f"f{fidx}",
            fontsize=7
        )
    cap.release()

plt.suptitle(
    f"Two-Speaker Face Crop Inspection — {n_tracks} Tracks\n"
    f"SUCCESS if you see 2 distinct faces across rows",
    fontsize=11, y=1.01
)
plt.tight_layout()
plt.savefig("demo/two_speaker_output/crop_visual_check.png", dpi=120, bbox_inches="tight")
plt.show()
from google.colab import files
files.download("demo/two_speaker_output/crop_visual_check.png")


---
# Cell 11: Structure TalkNet Output → Clean JSON (Task 00011)
Converts TalkNet's raw pickle outputs into `speaking_results.json` — a clean, per-frame schema with:
- `speaker_id` (track index)
- `timestamp_sec` (absolute time in source video)
- `speaking_probability` (0–1, smoothed ±2 frame window)
- `is_speaking` (boolean, probability > threshold)
- `bbox` ([x1, y1, x2, y2] in original video coordinates)

In [ ]:
# Cell 11: Upload talknet_to_json.py and generate speaking_results.json
import os
os.chdir("/content/TalkNet-ASD")

print("Upload talknet_to_json.py from your scripts/ml/ folder:")
from google.colab import files
uploaded = files.upload()
!mv talknet_to_json.py /content/TalkNet-ASD/

SAVE_DIR = "demo/test_clip"  # change to two_speaker_output for the two-speaker run

print("\n=== Structuring TalkNet Output to JSON ===")
!python talknet_to_json.py \
    --saveDir {SAVE_DIR} \
    --fps 25 \
    --threshold 0.5 \
    --output speaking_results.json

import json
with open(f"{SAVE_DIR}/speaking_results.json") as f:
    result = json.load(f)

print(f"\nSpeakers found: {len(result['speakers'])}")
for sp in result['speakers']:
    print(f"  Speaker {sp['speaker_id']}: {sp['start_time_sec']:.2f}s → {sp['end_time_sec']:.2f}s, {len(sp['frames'])} frames")

print("\n--- First 3 frames of Speaker 00000 ---")
print(json.dumps(result['speakers'][0]['frames'][:3], indent=2))

from google.colab import files
files.download(f"{SAVE_DIR}/speaking_results.json")
print("\n=== speaking_results.json downloaded ===")

---
# Cell 12: Validate speaking_results.json
Runs the automated JSON schema validation suite to confirm all fields, value ranges, and temporal ordering are correct.

In [ ]:
# Cell 12: Upload and run test_talknet_json.py validation suite
import os
os.chdir("/content/TalkNet-ASD")

print("Upload test_talknet_json.py from your scripts/ml/ folder:")
from google.colab import files
uploaded = files.upload()
!mv test_talknet_json.py /content/TalkNet-ASD/

SAVE_DIR = "demo/test_clip"
print("\n=== Running JSON Validation Suite ===")
!python test_talknet_json.py --jsonPath {SAVE_DIR}/speaking_results.json
